# GAS-BayesSHAP — remaining paper experiments (Q1 gap closure)

Gaussian-Adaptive Stratified Bayesian Shapley Estimation (v11.0). This notebook runs the large compute that completes the Q1-readiness review's evidence table: the **N=50 multi-instance sweep** (wine + air), the full matched-budget **curves**, the **width-vs-budget** curve, **Tier-B** group-lag at N=20, and the **R=500 coverage calibration**.

It orchestrates the real CLI entry points (`scripts/run_paper_experiments.py`, `scripts/coverage_validation.py`); it duplicates **no** scientific algorithm. Every output is written with a config-suffixed name (`*_n{N}_budget{B}.csv`) so prior runs are never overwritten, then copied into `main_results/`.

## 1. Environment

In [1]:
import sys, os, time, json, re, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
import gas_bayesshap

ROOT = Path("..").resolve()            # repo root (notebook runs from main_results/)
SCRIPTS = ROOT / "scripts"

print("GAS-BayesSHAP", gas_bayesshap.__version__)
print("repo root  :", ROOT)
wine_csv = ROOT / "data" / "winequality-white.csv"
air_csv  = ROOT / "data" / "Beijing_MultiSite_AirQuality.csv"
print("wine data  :", "OK" if wine_csv.exists() else "MISSING -> loader falls back to synthetic")
print("air  data  :", "OK" if air_csv.exists() else "MISSING -> loader falls back to synthetic")
print("numpy      :", np.__version__, "| pandas:", pd.__version__)

GAS-BayesSHAP 11.0.0
repo root  : /Users/mlouhichi/Desktop/GAS-BayesSHAP
wine data  : OK
air  data  : OK
numpy      : 2.5.2 | pandas: 3.0.5


## 2. Configuration

In [2]:
# Full-size defaults; override every value via env vars for a quick check:
#   GAS_N=2 GAS_BUDGET=200 GAS_TIERB_N=2 GAS_COV_TRIALS=10 GAS_SKIP=1
N          = int(os.environ.get("GAS_N", "50"))          # instances per dataset
EPS        = float(os.environ.get("GAS_EPS", "0.05"))    # tight epsilon
BUDGET     = int(os.environ.get("GAS_BUDGET", "3000"))   # coalition budget
TIERB_N    = int(os.environ.get("GAS_TIERB_N", "20"))    # Tier-B instances
COV_TRIALS = int(os.environ.get("GAS_COV_TRIALS", "500"))# calibration trials
SKIP_HEAVY = os.environ.get("GAS_SKIP", "0") == "1"     # skip curves/widths (smoke mode)

def run(*args, tag=""):
    """Run a CLI script with live output; raise on failure."""
    cmd = [sys.executable, str(SCRIPTS / args[0]), *args[1:]]
    t0 = time.time()
    print(f"\n>>> {tag or ' '.join(args)}")
    r = subprocess.run(cmd, cwd=ROOT)
    dt = time.time() - t0
    if r.returncode != 0:
        raise RuntimeError(f"FAILED ({r.returncode}): {' '.join(args)}")
    print(f"<<< done in {dt/60:.1f} min")
    return dt

print(f"N={N}  eps={EPS}  budget={BUDGET}  tierb_N={TIERB_N}  cov_trials={COV_TRIALS}  skip_heavy={SKIP_HEAVY}")

N=50  eps=0.05  budget=3000  tierb_N=20  cov_trials=500  skip_heavy=False


## 3. RQ1 — wine, N-instance tight-epsilon sweep

In [3]:
run("run_paper_experiments.py", "--only", "wine", "--n", str(N), "--eps", str(EPS), "--budget", str(BUDGET),
    tag=f"wine  N={N} eps={EPS} budget={BUDGET}")


>>> wine  N=50 eps=0.05 budget=3000
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000378 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 3428, number of used features: 11
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000349 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 3428, number of used features: 11
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147
[wine] surr

9753.009131908417

## 4. RQ2 — air (4 regimes), N-instance tight-epsilon sweep

In [4]:
run("run_paper_experiments.py", "--only", "air", "--n", str(N), "--eps", str(EPS), "--budget", str(BUDGET),
    tag=f"air   N={N} eps={EPS} budget={BUDGET}")


>>> air   N=50 eps=0.05 budget=3000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000493 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2163
[LightGBM] [Info] Number of data points in the train set: 22313, number of used features: 11
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000903 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2163
[LightGBM] [Info] Number of data points in the train set: 22313, number of used features: 11
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.3862

8713.355277061462

## 5. RQ2 — matched-budget curves (K = 128 … 2048)

GAS-BayesSHAP vs KernelSHAP vs SamplingSHAP at **identical** coalition budgets. Honest caveat from the committed N=8 curves: KernelSHAP overtakes GAS at K ≥ 1024 on these low-dimensional (M=11) games; GAS wins at low K (≤ 512) where its Bayesian control variate pays off.

In [5]:
if SKIP_HEAVY:
    print("SKIPPED (GAS_SKIP=1) — full curve sweep is ~1 h on a laptop.")
else:
    run("run_paper_experiments.py", "--only", "curves", tag="matched-budget curves")


>>> matched-budget curves
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000408 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 3428, number of used features: 11
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000392 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 3428, number of used features: 11
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147
  K=128: gas=0.00430 

## 6. RQ1 — certificate width vs budget

In [6]:
if SKIP_HEAVY:
    print("SKIPPED (GAS_SKIP=1) — width-vs-budget is ~30 min on a laptop.")
else:
    run("run_paper_experiments.py", "--only", "widths", tag="width-vs-budget")


>>> width-vs-budget
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000311 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 3428, number of used features: 11
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000282 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1295
[LightGBM] [Info] Number of data points in the train set: 3428, number of used features: 11
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147
[wine] widths-vs-budget (su

## 7. RQ3 — Tier-B group-lag game (M = 66 → 11 pollutant macros)

Group-lag Shapley over 11 pollutant macros (6 lags each) for the air-quality task; macro simultaneous coverage is reported. N is no longer capped at 10.

In [7]:
run("run_paper_experiments.py", "--only", "tierb", "--n", str(TIERB_N),
    tag=f"tierb N={TIERB_N} eps={EPS} budget={BUDGET}")


>>> tierb N=20 eps=0.05 budget=3000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002056 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12962
[LightGBM] [Info] Number of data points in the train set: 22296, number of used features: 66
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[tierB] lagged surrogate acc=0.9757
[tierB] summary: rmse=0.00202 sim_cov=1.00 sign_cert=0.000 mean_width=9.02

done in 916s; results in results/paper_experiments/ and main_results/paper_*.csv
<<< done in 15.3 min


918.8210318088531

## 8. Calibration — coverage validation (R repeated trials)

Repeated synthetic trials (M=3 calibration game) to check the anytime empirical-Bernstein certificate: finite-width rate, empirical coverage, coverage-given-finite. The report is parsed from the CLI output and persisted as JSON next to the other paper artifacts. The real-data counterpart is the simultaneous-coverage rate of the N-sweeps above.

In [8]:
cmd = [sys.executable, str(SCRIPTS / "coverage_validation.py"),
       "--trials", str(COV_TRIALS), "--M", "3", "--epsilon", "1.5",
       "--delta", "0.05", "--max-budget", "300"]
t0 = time.time()
r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(r.stdout)
if r.stderr.strip():
    print(r.stderr)
if r.returncode != 0:
    raise RuntimeError(f"coverage_validation failed ({r.returncode})")

# --- persist the printed report as JSON (same schema as paper_coverage_calibration_R500.json)
KEYS = ("n_trials", "finite_width_rate", "empirical_coverage",
        "coverage_given_finite", "mean_width", "median_width", "max_width",
        "oracle_query_cost (mean)", "oracle_query_cost (max)")
rep = {}
for line in r.stdout.splitlines():
    m = re.match(r"^\s*(.*?)\s*:\s*(.+?)\s*$", line)
    if m and m.group(1) in KEYS:
        key = m.group(1).replace(" ", "_").replace("(", "").replace(")", "")
        val = m.group(2)
        try:
            rep[key] = float(val) if "." in val else int(val)
        except ValueError:
            rep[key] = val
payload = {"n_trials": COV_TRIALS, "M": 3, "epsilon": 1.5, "delta": 0.05,
           "max_budget": 300, **rep}
for dest in (ROOT / "results" / "paper_experiments", ROOT / "main_results"):
    dest.mkdir(parents=True, exist_ok=True)
    (dest / f"paper_coverage_calibration_R{COV_TRIALS}.json").write_text(
        json.dumps(payload, indent=1))
print(f"persisted paper_coverage_calibration_R{COV_TRIALS}.json in "
      f"results/paper_experiments/ and main_results/ ({time.time()-t0:.0f}s)")

=== Coverage validation ===
  n_trials                : 500
  finite_width_rate       : 1.0000
  empirical_coverage      : 1.0000
  coverage_given_finite   : 1.0000
  mean_width              : 12.3060
  median_width            : 12.2855
  max_width               : 12.6113
  oracle_query_cost (mean): 350.0
  oracle_query_cost (max) : 350

persisted paper_coverage_calibration_R500.json in results/paper_experiments/ and main_results/ (62s)


## 9. Summary of produced artifacts

In [9]:
def show(name, path):
    p = Path(path)
    if p.exists():
        print(f"\n[{name}]")
        print(pd.read_csv(p).to_string(index=False))
    else:
        print(f"\n[{name}] NOT FOUND: {p.name}")

show("WINE N-sweep",  ROOT / "main_results" / f"paper_wine_n{N}_budget{BUDGET}_summary.csv")
show("AIR  N-sweep",  ROOT / "main_results" / f"paper_air_n{N}_budget{BUDGET}_summary.csv")
show("TIER-B",        ROOT / "main_results" / "paper_air_tierB_summary.csv")
show("WINE matched-budget", ROOT / "main_results" / "paper_wine_matched_budget.csv")
show("AIR  matched-budget", ROOT / "main_results" / "paper_air_matched_budget.csv")
cov = ROOT / "main_results" / f"paper_coverage_calibration_R{COV_TRIALS}.json"
if cov.exists():
    print(f"\n[COVERAGE CALIBRATION R={COV_TRIALS}]")
    print(json.dumps(json.loads(cov.read_text()), indent=1))


[WINE N-sweep]
dataset  n_instances  clusters  eps  budget  rmse_gas_mean  rmse_gas_std  rmse_kernel_mean  rmse_mc_mean  simultaneous_coverage_rate  marginal_coverage_rate  sign_certified_fraction  mean_width  max_width_max  gas_evals_mean  exact_evals  kernel_evals  mc_evals_mean  converged_fraction            status_counts
   wine           50         2 0.05    3000        0.00168       0.00101          0.004715       0.04655                         1.0                     1.0                      0.0    9.147107       15.37875          1430.3         2048           256        1629.96                 0.0 {'BUDGET_EXHAUSTED': 50}

[AIR  N-sweep]
dataset  n_instances  clusters  eps  budget  rmse_gas_mean  rmse_gas_std  rmse_kernel_mean  rmse_mc_mean  simultaneous_coverage_rate  marginal_coverage_rate  sign_certified_fraction  mean_width  max_width_max  gas_evals_mean  exact_evals  kernel_evals  mc_evals_mean  converged_fraction            status_counts
    air           50         4 0

## Notes — expected runtime and honest caveats

- **Measured cost** (sandbox, wine): ≈ 20 ms / coalition-eval ⇒ a budget-3000 instance ≈ 100 s. Full default run ≈ 3.5–4 h on a laptop: wine N=50 ≈ 1.5 h, air N=50 ≈ 1.5 h, curves ≈ 1 h, widths ≈ 30 min, Tier-B N=20 ≈ 30 min, R=500 ≈ 10–20 min.
- **Certificates stay conservative**: past runs give mean certified width ≈ 9 vs attribution scale ≈ 0.3, hence `sign_certified_fraction = 0` and `converged_fraction = 0` (status `BUDGET_EXHAUSTED`) — the intervals are valid but wide. This is the paper's open problem; the width-tightening (`range_mode="empirical_max"`, flagged `range_bound_is_heuristic=True`) is the opt-in mitigation.
- **Curves**: KernelSHAP overtakes GAS at K ≥ 1024 on M=11 games; the claim is that GAS wins in the low-budget regime (K ≤ 512), not that it dominates everywhere.
- **Coverage**: the R=500 calibration validates the certificate *mechanism* on a synthetic game; the N-sweep simultaneous-coverage rates carry that check to real data.
- All artifacts land in `results/paper_experiments/` and are copied to `main_results/` with the `paper_` prefix; commit them with the notebook when the run completes.